# Primewords 中文测试集评估

在 primewords 100 条短音频测试集上，评估 **微调前后** SenseVoice 和 Fun-ASR-Nano 的 CER 变化。

**测试集**: `local/primewords_test.jsonl` - 100 条最短音频（平均 2.5s）
**模型**: SenseVoice-base / SenseVoice-ft / Nano-base / Nano-ft

## 1. 环境检查

In [1]:
import warnings
import logging
logging.getLogger('root').setLevel(logging.ERROR)
warnings.filterwarnings('ignore')

import os, re, json, time, gc
import torch

device = 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'PyTorch: {torch.__version__}')
print(f'Device: {device}')

import funasr
print(f'FunASR: {funasr.__version__}')

import editdistance
print('editdistance: OK')

print('\n--- 路径检查 ---')
test_jsonl = '/Users/fanhua/Projects/Agent/local/primewords_test.jsonl'
print(f'  测试集: {"OK" if os.path.exists(test_jsonl) else "NOT FOUND"}')

with open(test_jsonl) as f:
    samples = [json.loads(line) for line in f]
exists = sum(1 for s in samples if os.path.exists(s['audio_filepath']))
print(f'  音频: {exists}/{len(samples)} 条存在')

SV_CKPT = os.path.expanduser('~/Projects/Agent/model/sensevoice_lora/model.pt.best')
NANO_CKPT = os.path.expanduser('~/Projects/Agent/model/funasr_nano_v3/model.pt.best')
print(f'  SenseVoice-ft: {"OK" if os.path.exists(SV_CKPT) else "NOT FOUND"}')
print(f'  Nano-ft: {"OK" if os.path.exists(NANO_CKPT) else "NOT FOUND"}')

PyTorch: 2.11.0
Device: mps


FunASR: 1.3.1
editdistance: OK

--- 路径检查 ---
  测试集: OK
  音频: 100/100 条存在
  SenseVoice-ft: OK
  Nano-ft: OK


## 2. 工具函数

In [2]:
def clean_sensevoice_text(text):
    return re.sub(r'<\|[^|]*\|>', '', text).strip()

_PUNCT = re.compile(
    r'[\s\.,!?;:\-()\[\]{}\u3000\uff0c\u3002\uff01\uff1f\u3001\uff1b\uff1a'
    r'\u201c\u201d\u2018\u2019\uff08\uff09\u3010\u3011\u300a\u300b\u2026\u2014]'
)

def norm_punct(text):
    return _PUNCT.sub('', text)

def levenshtein(s1, s2):
    if len(s1) < len(s2):
        return levenshtein(s2, s1)
    if len(s2) == 0:
        return len(s1)
    prev = list(range(len(s2) + 1))
    for c1 in s1:
        curr = [prev[0] + 1]
        for j, c2 in enumerate(s2):
            curr.append(min(prev[j + 1] + 1, curr[j] + 1, prev[j] + (c1 != c2)))
        prev = curr
    return prev[-1]

def free_memory():
    gc.collect()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()

print('工具函数就绪')

工具函数就绪


## 3. 加载测试集

In [3]:
TEST_JSONL = '/Users/fanhua/Projects/Agent/local/primewords_test.jsonl'

samples = []
with open(TEST_JSONL, 'r', encoding='utf-8') as f:
    for line in f:
        s = json.loads(line)
        if os.path.exists(s['audio_filepath']):
            samples.append(s)

print(f'测试集: {len(samples)} 条')
lengths = [s['length'] for s in samples]
print(f'长度: min={min(lengths):.2f}s max={max(lengths):.2f}s avg={sum(lengths)/len(lengths):.2f}s')

print(f'\n前5条:')
for s in samples[:5]:
    print(f'  {os.path.basename(s["audio_filepath"])} -> {s["text"]}')

测试集: 100 条
长度: min=1.32s max=2.78s avg=2.49s

前5条:
  d9f16824-ab67-4088-ba2c-8449c3fef822.wav -> 叶 旅 
  2162f1f7-eacf-428b-ab3d-962b430c2319.wav -> 五 
  7f00cb9e-9d8d-4ed9-8733-bdeb1606f158.wav ->  十五料理 
  33a9e17c-8eed-4edb-874f-cb49fff86275.wav ->  内魔法 同上
  9def61e6-0ac1-4b0d-858d-ec24be06aee0.wav ->  一人 歩 


## 4. 评估函数

In [4]:
from funasr import AutoModel

def eval_sensevoice(model, samples, label):
    results = []
    total_cer, total_chars, exact = 0.0, 0, 0
    start = time.time()
    
    for i, s in enumerate(samples):
        audio = s['audio_filepath']
        expected_raw = s['text']
        if not os.path.exists(audio):
            continue
        try:
            res = model.generate(input=audio, language='auto', use_itn=True)
            pred_raw = clean_sensevoice_text(res[0]['text']) if res else ''
        except:
            pred_raw = ''
        
        expected = norm_punct(expected_raw)
        pred = norm_punct(pred_raw)
        dist = levenshtein(expected, pred)
        cer = dist / max(len(expected), 1)
        total_cer += dist
        total_chars += max(len(expected), 1)
        if expected == pred:
            exact += 1
        
        results.append({
            'id': i, 'expected_raw': expected_raw, 'predicted_raw': pred_raw,
            'expected': expected, 'predicted': pred, 'cer': cer
        })
        
        if (i + 1) % 20 == 0:
            print(f'    {label}: {i+1}/{len(samples)} | CER={total_cer/total_chars:.2%} | exact={exact}')
    
    return {
        'name': label, 'results': results,
        'cer': total_cer/total_chars if total_chars > 0 else 0,
        'exact': exact, 'total': len(results), 'time': time.time() - start
    }

def eval_nano(model, samples, label):
    results = []
    total_cer, total_chars, exact = 0.0, 0, 0
    start = time.time()
    
    with torch.inference_mode():
        for i, s in enumerate(samples):
            audio = s['audio_filepath']
            expected_raw = s['text']
            if not os.path.exists(audio):
                continue
            try:
                res = model.generate(input=audio, language='中文', itn=True)
                pred_raw = res[0]['text'] if res else ''
            except:
                pred_raw = ''
            
            expected = norm_punct(expected_raw)
            pred = norm_punct(pred_raw)
            dist = levenshtein(expected, pred)
            cer = dist / max(len(expected), 1)
            total_cer += dist
            total_chars += max(len(expected), 1)
            if expected == pred:
                exact += 1
            
            results.append({
                'id': i, 'expected_raw': expected_raw, 'predicted_raw': pred_raw,
                'expected': expected, 'predicted': pred, 'cer': cer
            })
            
            if (i + 1) % 20 == 0:
                print(f'    {label}: {i+1}/{len(samples)} | CER={total_cer/total_chars:.2%} | exact={exact}')
    
    return {
        'name': label, 'results': results,
        'cer': total_cer/total_chars if total_chars > 0 else 0,
        'exact': exact, 'total': len(results), 'time': time.time() - start
    }

print('评估函数就绪')

评估函数就绪


## 5. 评估 [1/4] SenseVoice-base (未微调)

In [5]:
print('[1/4] SenseVoice-base (未微调)...')
start = time.time()

sv_model = AutoModel(model='iic/SenseVoiceSmall', disable_update=True, device=device)
print(f'模型加载完成, 耗时: {time.time() - start:.1f}s')

sv_base_res = eval_sensevoice(sv_model, samples, 'SV-base')
print(f'\n完成! CER={sv_base_res["cer"]:.2%} | exact={sv_base_res["exact"]}/{sv_base_res["total"]} | {sv_base_res["time"]:.1f}s')

del sv_model
free_memory()

[1/4] SenseVoice-base (未微调)...
funasr version: 1.3.1.


模型加载完成, 耗时: 3.2s


  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00,  3.13it/s]

{'load_data': '0.049', 'extract_feat': '0.008', 'forward': '0.320', 'batch_size': '1', 'rtf': '0.242'}, : 100%|██████████| 1/1 [00:00<00:00,  3.13it/s]

...

  0%|          | 0/1 [00:00<?, ?it/s]

{'load_data': '0.056', 'extract_feat': '0.003', 'forward': '0.094', 'batch_size': '1', 'rtf': '0.034'}, : 100%|██████████| 1/1 [00:00<00:00, 10.60it/s]

rtf_avg: 0.034: 100%|██████████| 1/1 [00:00<00:00, 10.55it/s]                                                                                          

rtf_avg: 0.034: 100%|██████████| 1/1 [00:00<00:00, 10.52it/s]

    SV-base: 100/100 | CER=15.95% | exact=38

完成! CER=15.95% | exact=38/100 | 12.7s


## 6. 评估 [2/4] SenseVoice-ft (微调后)

In [6]:
print('[2/4] SenseVoice-ft (微调)...')
start = time.time()

sv_ft_model = AutoModel(model='iic/SenseVoiceSmall', lora_only=True, disable_update=True, device=device)
ckpt = torch.load(SV_CKPT, map_location='cpu', weights_only=False)
keys_matched = len(set(sv_ft_model.model.state_dict().keys()) & set(ckpt['state_dict'].keys()))
sv_ft_model.model.load_state_dict(ckpt['state_dict'], strict=False)
del ckpt
print(f'微调权重已载入 (keys: {keys_matched}), 耗时: {time.time() - start:.1f}s')

sv_ft_res = eval_sensevoice(sv_ft_model, samples, 'SV-ft')
print(f'\n完成! CER={sv_ft_res["cer"]:.2%} | exact={sv_ft_res["exact"]}/{sv_ft_res["total"]} | {sv_ft_res["time"]:.1f}s')

del sv_ft_model
free_memory()

[2/4] SenseVoice-ft (微调)...
funasr version: 1.3.1.


微调权重已载入 (keys: 917), 耗时: 4.4s


  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00,  4.46it/s]

{'load_data': '0.139', 'extract_feat': '0.018', 'forward': '0.224', 'batch_size': '1', 'rtf': '0.170'}, : 100%|██████████| 1/1 [00:00<00:00,  4.46it/s]

...

  0%|          | 0/1 [00:00<?, ?it/s]

{'load_data': '0.056', 'extract_feat': '0.003', 'forward': '0.095', 'batch_size': '1', 'rtf': '0.034'}, : 100%|██████████| 1/1 [00:00<00:00, 10.53it/s]

rtf_avg: 0.034: 100%|██████████| 1/1 [00:00<00:00, 10.46it/s]                                                                                          

rtf_avg: 0.034: 100%|██████████| 1/1 [00:00<00:00, 10.43it/s]

    SV-ft: 100/100 | CER=30.45% | exact=16

完成! CER=30.45% | exact=16/100 | 12.3s


## 7. 评估 [3/4] Nano-base (未微调)

In [7]:
print('[3/4] Nano-base (未微调)...')
start = time.time()

nano_model = AutoModel(model='FunAudioLLM/Fun-ASR-Nano-2512', hub='ms', device=device, disable_update=True)
print(f'模型加载完成, 耗时: {time.time() - start:.1f}s')

nano_base_res = eval_nano(nano_model, samples, 'Nano-base')
print(f'\n完成! CER={nano_base_res["cer"]:.2%} | exact={nano_base_res["exact"]}/{nano_base_res["total"]} | {nano_base_res["time"]:.1f}s')

del nano_model
free_memory()

[3/4] Nano-base (未微调)...
funasr version: 1.3.1.


模型加载完成, 耗时: 13.5s


  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00,  1.54it/s]

{'load_data': '0.075', 'extract_feat': '0.002', 'forward': '0.651', 'batch_size': '1', 'rtf': '0.493'}, : 100%|██████████| 1/1 [00:00<00:00,  1.54it/s]

...

100%|██████████| 1/1 [00:00<00:00,  2.42it/s]

{'load_data': '0.053', 'extract_feat': '0.002', 'forward': '0.414', 'batch_size': '1', 'rtf': '0.150'}, : 100%|██████████| 1/1 [00:00<00:00,  2.42it/s]

rtf_avg: 0.150: 100%|██████████| 1/1 [00:00<00:00,  2.42it/s]                                                                                          

rtf_avg: 0.150: 100%|██████████| 1/1 [00:00<00:00,  2.41it/s]

    Nano-base: 100/100 | CER=7.68% | exact=63

完成! CER=7.68% | exact=63/100 | 91.4s


## 8. 评估 [4/4] Nano-ft (微调后)

In [8]:
print('[4/4] Nano-ft (微调)...')
start = time.time()

nano_ft_model = AutoModel(
    model='FunAudioLLM/Fun-ASR-Nano-2512', hub='ms', device=device, disable_update=True,
    init_param=NANO_CKPT,
)
print(f'微调权重已载入, 耗时: {time.time() - start:.1f}s')

nano_ft_res = eval_nano(nano_ft_model, samples, 'Nano-ft')
print(f'\n完成! CER={nano_ft_res["cer"]:.2%} | exact={nano_ft_res["exact"]}/{nano_ft_res["total"]} | {nano_ft_res["time"]:.1f}s')

del nano_ft_model
free_memory()

[4/4] Nano-ft (微调)...
funasr version: 1.3.1.


微调权重已载入, 耗时: 33.0s


  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00,  1.99it/s]

{'load_data': '0.135', 'extract_feat': '0.022', 'forward': '0.501', 'batch_size': '1', 'rtf': '0.380'}, : 100%|██████████| 1/1 [00:00<00:00,  1.99it/s]

...

100%|██████████| 1/1 [00:00<00:00,  2.43it/s]

{'load_data': '0.060', 'extract_feat': '0.002', 'forward': '0.412', 'batch_size': '1', 'rtf': '0.149'}, : 100%|██████████| 1/1 [00:00<00:00,  2.43it/s]

rtf_avg: 0.149: 100%|██████████| 1/1 [00:00<00:00,  2.43it/s]                                                                                          

rtf_avg: 0.149: 100%|██████████| 1/1 [00:00<00:00,  2.42it/s]

    Nano-ft: 100/100 | CER=9.80% | exact=57

完成! CER=9.80% | exact=57/100 | 93.0s


## 9. 结果汇总

In [9]:
all_res = [sv_base_res, sv_ft_res, nano_base_res, nano_ft_res]
names = [r['name'] for r in all_res]
cers = [r['cer'] for r in all_res]
exact_rates = [r['exact'] / max(r['total'], 1) for r in all_res]

sv_imp = (sv_base_res['cer'] - sv_ft_res['cer']) / sv_base_res['cer'] * 100
nano_imp = (nano_base_res['cer'] - nano_ft_res['cer']) / nano_base_res['cer'] * 100

print('=' * 65)
print('Primewords 测试集: 微调前后中文 ASR 对比 (100 条短音频)')
print('=' * 65)
print(f'{"":<18}' + ''.join(f'{n:>12}' for n in names))
print('-' * 65)
print(f'{"CER":<18}' + ''.join(f'{c:>11.2%}' for c in cers))
print(f'{"精确匹配率":<18}' + ''.join(f'{e:>11.1%}' for e in exact_rates))
print('-' * 65)
print(f'{"微调相对提升":<18}' + f'{sv_imp:>+11.1f}%' + ' ' * 12 + f'{nano_imp:>+10.1f}%')
print('=' * 65)

Primewords 测试集: 微调前后中文 ASR 对比 (100 条短音频)
                       SV-base       SV-ft   Nano-base     Nano-ft
-----------------------------------------------------------------
CER                    15.95%     30.45%      7.68%      9.80%
精确匹配率                   38.0%      16.0%      63.0%      57.0%
-----------------------------------------------------------------
微调相对提升                  -91.0%                 -27.5%


## 10. 可视化

In [10]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
matplotlib.rcParams['font.family'] = ['-apple-system', 'PingFang SC', 'Microsoft YaHei', 'sans-serif']
matplotlib.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

model_labels = ['SenseVoice\n-base', 'SenseVoice\n-ft', 'Nano\n-base', 'Nano\n-ft']
colors = ['#e74c3c', '#27ae60', '#e67e22', '#3498db']

# CER
ax = axes[0]
bars = ax.bar(model_labels, [c * 100 for c in cers], color=colors, width=0.5)
ax.set_ylabel('CER (%)', fontsize=12)
ax.set_title('CER 对比 (越低越好)', fontsize=13, fontweight='bold')
ax.set_ylim(0, max(cers) * 100 * 1.4)
for bar, c in zip(bars, cers):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{c*100:.1f}%',
            ha='center', va='bottom', fontsize=11)

# Exact match rate
ax = axes[1]
bars = ax.bar(model_labels, [e * 100 for e in exact_rates], color=colors, width=0.5)
ax.set_ylabel('精确匹配率 (%)', fontsize=12)
ax.set_title('精确匹配率对比 (越高越好)', fontsize=13, fontweight='bold')
ax.set_ylim(0, 100)
for bar, e in zip(bars, exact_rates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{e*100:.1f}%',
            ha='center', va='bottom', fontsize=11)

fig.suptitle('Primewords 测试集: 微调前后中文 ASR 性能对比', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/Users/fanhua/Projects/Agent/local/primewords_eval.png', dpi=150, bbox_inches='tight')
plt.show()

print('图表已保存: /Users/fanhua/Projects/Agent/local/primewords_eval.png')

图表已保存: /Users/fanhua/Projects/Agent/local/primewords_eval.png


## 11. 保存结果

In [11]:
save_path = '/Users/fanhua/Projects/Agent/local/primewords_eval_result.json'
with open(save_path, 'w', encoding='utf-8') as f:
    json.dump({
        'test_set': 'primewords_100_short',
        'total_samples': len(samples),
        'models': {
            r['name']: {
                'cer': round(r['cer'], 4),
                'exact': r['exact'],
                'total': r['total'],
                'exact_rate': round(r['exact'] / max(r['total'], 1), 4),
                'time': round(r['time'], 1),
            } for r in all_res
        },
        'improvement': {
            'SenseVoice (微调提升)': f'{sv_imp:+.1f}%',
            'Nano (微调提升)': f'{nano_imp:+.1f}%',
        }
    }, f, ensure_ascii=False, indent=2)

print(f'结果已保存: {save_path}')

结果已保存: /Users/fanhua/Projects/Agent/local/primewords_eval_result.json
